In [ ]:
import dapla as dp
import pandas as pd
import timeseries as ts
from timeseries.dataset import Dataset
from timeseries.properties import SeriesType, Versioning, Temporality
from timeseries import dates
from datetime import datetime
import pytz
#with ProjectRoot():
#    import src.timeseries as ts
#    from src.timeseries.dataset import Dataset

In [ ]:
TIMEZONE = 'Europe/Oslo'
timeformat = "%Y-%m-%dT%H:%M:%S%z"
TZ_NORWAY = pytz.timezone(TIMEZONE)
datetime.now(TZ_NORWAY).strftime(timeformat)

In [ ]:
tz_nor

In [ ]:
local_dt = tz_nor.localize(datetime.now(), is_dst=None)
print(local_dt.strftime(timeformat))
utc_dt = local_dt.astimezone(pytz.utc)
print(utc_dt.strftime(timeformat))

In [ ]:
tz_nor.localize(datetime.now(), is_dst=None).strftime(timeformat)

In [ ]:
tz_nor.localize(datetime.now(), is_dst=None).astimezone(pytz.utc).strftime(timeformat)

In [ ]:
pytz.utc

In [ ]:
datetime(2024, 2, 11, 15, 0, 0, tzinfo=pytz.utc).strftime(timeformat)

In [ ]:
datetime.utcnow().strftime(timeformat)

In [ ]:
 # Current time in UTC
now_utc = datetime.now(pytz.timezone('UTC'))
print(now_utc.strftime(timeformat))
 
# Convert to Asia/Kolkata time zone
now_asia = now_utc.astimezone(pytz.timezone(TIMEZONE))
print(now_asia.strftime(timeformat))

In [ ]:
from datetime import datetime, timezone
datetime.now(timezone.utc).strftime(timeformat)

In [ ]:
from datetime import datetime, timezone
datetime.now(timezone.utc).strftime(timeformat)

In [ ]:
dates.date_utc(datetime.now()).strftime(timeformat)

In [ ]:
dates.date_utc(datetime(2023, 6, 1, 9, 0, 0)).strftime("%Y-%m-%dT%H:%M:%S%z")

In [ ]:
offbal_df = dp.read_pandas(
    "gs://ssb-prod-finregn-data-produkt/inndata/offentlig/offbal_test.parquet",
    file_format="parquet",
)

In [ ]:
serie_cols = [
    "sek_esa",
    "acc_entry",
    "objekt",
    "flow_stock",
    "func_cat",
    "maturity",
    "land",
]

offbal_df["serienavn"] = offbal_df[serie_cols].apply(lambda row: ".".join(row), axis=1)

offbal_tid = (
    offbal_df[["new_datetime", "serienavn", "verdi"]]
    .pivot(index="new_datetime", columns="serienavn")
    .fillna(0)
    .droplevel(0, axis=1)
)

offbal_tid.columns.name = None
offbal_tid.index = pd.PeriodIndex(offbal_tid.index, freq="Q")
offbal_tid.index.name = "periode"
offbal_tid = offbal_tid.reset_index()#.set_index('new_datetime', drop=True)  # .transpose()
offbal_tid

In [ ]:
datalist = []
for year in sorted(list(set([p.split('Q')[0] for p in offbal_tid.periode.astype(str).tolist()]))):
    d = offbal_tid[(offbal_tid.periode >= f"{year}Q1") & (offbal_tid.periode <= f"{year}Q4")]
    datalist.append(d)

In [ ]:
startdata = pd.concat(datalist[:-3])

In [ ]:
datadict = {}
for year in sorted(list(set([p.split('Q')[0] for p in offbal_tid.periode.astype(str).tolist()]))):
    datadict[year] = offbal_tid[(offbal_tid.periode >= f"{year}Q1") & (offbal_tid.periode <= f"{year}Q4")]

In [ ]:
data21 = datadict['2021']
data22 = datadict['2022']
data23 = datadict['2023']

## Init

In [ ]:
tsname = "test-offbal"
seriestype = SeriesType.as_of_at()
#seriestype = SeriesType.none_at()

In [ ]:
asof_start = datetime(2020, 12, 30)

In [ ]:
startingdata = Dataset(
    name=tsname,
    data_type=seriestype,
    as_of_tz=asof_start,
    data=startdata,
)

startingdata

In [ ]:
startingdata.data

## Save

In [ ]:
startingdata.save()

## Read

In [ ]:
readdata = Dataset(
    name=tsname, data_type=seriestype, as_of_tz=asof_start
)
readdata

In [ ]:
readdata.data

## Append

Mulig å appendere hvis SeriesType.none_at(), men da er det ingen versjonering av datasett...

In [ ]:
asof_21 = "2021-12-30"

In [ ]:
data2021 = Dataset(
    name=tsname,
    data_type=seriestype,
    as_of_tz=asof_21,
    data=data21,
)

data2021

In [ ]:
data2021.data

In [ ]:
data2021.save()

In [ ]:
Dataset(
    name=tsname,
    data_type=seriestype,
    as_of_tz=asof_21,
    #data=data21,
).data

In [ ]:
Dataset(
    name=tsname
)